# 05a — Starting advantage: road + third settlement
The first two placement rounds are the only game data used. Calculate the probability of having 2 wood, 2 brick, 1 sheep and 1 wheat after any valid 4:1 trades, assuming no trading with players or ports, fixed production, no robber, no discard, and unlimited bank resources. Each trade uses four identical surplus cards; any missing required card may be acquired, even if that resource is produced directly. Ore is tradeable. No actual game actions or dice outcomes are used.

The state transition uses the complete joint production vector for each dice sum and its probability. An absorbing Markov chain accumulates readiness without rolling dice. Surplus caps preserve the decision because at most six required cards can be missing. For player p, own turn t occurs after 4(t−1)+p total dice rolls. A frozen board must have a legal settlement site reachable by one extra road. If not, construction readiness is zero and the position is Blocked. The target is the first eligible personal turn with probability ≥50%; 40 turns is the search horizon, not a maximum possible time.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from advantage_model import build_advantage, affordable, readiness_curve, MAX_TURNS
OUT = Path('processed_data_5000')
# Sanity checks: trade four identical surplus cards; cannot mix two types.
assert affordable((6,2,1,0,0))
assert not affordable((4,2,1,0,2))
assert affordable((0,0,0,0,24))
assert not affordable((0,0,0,0,23))
assert not affordable((6,2,1,0,0),trade=False)
zero=np.zeros((13,5),dtype=int)
assert np.allclose(readiness_curve((0,0,0,0,0),zero,8),0)
assert np.allclose(readiness_curve((2,2,1,1,0),zero,8),1)
# Only initial map and placements.
p, t, f = [pd.read_csv(OUT / n) for n in ('initial_placements.csv','map_tiles.csv','initial_features.csv')]
d, curves = build_advantage(p,t,f)
assert len(d) == len(p) and len(curves) == len(p)*MAX_TURNS
assert d.Player_Number.eq(d.Placement_Order).all()
assert d.Round_2_Order.eq(9-d.Player_Number).all()
assert d.loc[d.Blocked,'Turns_To_50'].isna().all()
d.to_csv(OUT/'starting_advantage.csv',index=False)
curves.to_csv(OUT/'readiness_curves.csv',index=False)
display(d[['Game_ID','Player','Player_Number','Round_1_Order','Round_2_Order','Expansion_Options','Turns_To_50','Status']].head())
display(d.Status.value_counts())
display(d.Turns_To_50.describe())

In [ ]:
import matplotlib.pyplot as plt
avg = curves.groupby('Personal_Turn')[['Build_Probability_Trade','Build_Probability_NoTrade']].mean()
(avg*100).plot(); plt.ylabel('Mean probability (%)'); plt.ylim(0,100);plt.title('Effect of 4:1 trading');plt.show()
by_order = curves.pivot_table(index='Personal_Turn',columns='Player_Number',values='Build_Probability_Trade')
(by_order*100).plot();plt.ylabel('Mean probability (%)');plt.ylim(0,100);plt.title('Readiness by player order');plt.show()
display(d.groupby('Player_Number')[['Turns_To_50','Prob_Turn_5','Prob_Turn_10','Blocked']].agg(['mean','median']))

Historical pilot results are not current results. Run this notebook on the new 5,000-game dataset before drawing conclusions.